charger tous les modele + prédiction

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/FakeNewsProject'

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np

BASE = '/content/drive/MyDrive/FakeNewsProject'

test = pd.read_csv(f'{BASE}/data/processed/test.csv')

X_test = test['clean_text'].fillna('')
y_test = test['label']

In [ ]:
import numpy as np
import pandas as pd

BASE = '/content/drive/MyDrive/FakeNewsProject'

# charger données test
test = pd.read_csv(f'{BASE}/data/processed/test.csv')

X_test = test['clean_text'].fillna('')
y_test = test['label']

# charger modèles déjà sauvegardés
lr_probs   = np.load(f'{BASE}/preds/lr_probs.npy')
lstm_probs = np.load(f'{BASE}/preds/lstm_probs.npy')
bert_probs = np.load(f'{BASE}/preds/bert_probs.npy')

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, roc_auc_score

BASE = '/content/drive/MyDrive/FakeNewsProject'

# RECHARGEMENT OBLIGATOIRE
y_test = np.load(f'{BASE}/data/y_test.npy')

lr_probs   = np.load(f'{BASE}/preds/lr_probs.npy')
lstm_probs = np.load(f'{BASE}/preds/lstm_probs.npy')
bert_probs = np.load(f'{BASE}/preds/bert_probs.npy')

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

BASE = '/content/drive/MyDrive/FakeNewsProject'

# charger données
test = pd.read_csv(f'{BASE}/data/processed/test.csv')

X_test = test['clean_text'].fillna('')
y_test = test['label']

# RECRÉER TF-IDF (OBLIGATOIRE)
train = pd.read_csv(f'{BASE}/data/processed/train.csv')
X_train = train['clean_text'].fillna('')
y_train = train['label']

tfidf = TfidfVectorizer(max_features=100000, ngram_range=(1,2),
                        sublinear_tf=True, min_df=2)

X_tr = tfidf.fit_transform(X_train)

# RECRÉER LOGISTIC REGRESSION
lr = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr.fit(X_tr, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [ ]:
!pip -q install transformers torch shap lime

from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/FakeNewsProject'

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

#  Vérifier et utiliser GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device utilisé :", device)

# Charger données
test = pd.read_csv(f'{BASE}/data/processed/test.csv')
y_test = test['label'].values

# Charger BERT
tok_bert = AutoTokenizer.from_pretrained(f'{BASE}/models/bert_final')
mdl_bert = AutoModelForSequenceClassification.from_pretrained(
    f'{BASE}/models/bert_final'
).to(device).eval()

# Fonction prédiction
def bert_predict_proba(texts):
    """Retourne probabilité FAKE pour chaque texte."""
    enc = tok_bert(
        texts,
        truncation=True,
        padding=True,
        max_length=512,
        return_tensors='pt'
    )

    #  envoyer les données au GPU
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        logits = mdl_bert(**enc).logits

    #  ramener résultat vers CPU pour numpy
    return torch.softmax(logits, dim=-1)[:, 0].cpu().numpy()

# Texte de test
texts_test = test['clean_text'].fillna('').tolist()

# Prédictions par batch
bert_probs = []

for i in range(0, len(texts_test), 32):
    print(f"Batch {i}/{len(texts_test)}")  # suivi progression
    bert_probs.extend(bert_predict_proba(texts_test[i:i+32]))

bert_probs = np.array(bert_probs)

print(f"✓ BERT prédictions : {bert_probs.shape}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device utilisé : cuda


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Batch 0/8980
Batch 32/8980
Batch 64/8980
Batch 96/8980
Batch 128/8980
Batch 160/8980
Batch 192/8980
Batch 224/8980
Batch 256/8980
Batch 288/8980
Batch 320/8980
Batch 352/8980
Batch 384/8980
Batch 416/8980
Batch 448/8980
Batch 480/8980
Batch 512/8980
Batch 544/8980
Batch 576/8980
Batch 608/8980
Batch 640/8980
Batch 672/8980
Batch 704/8980
Batch 736/8980
Batch 768/8980
Batch 800/8980
Batch 832/8980
Batch 864/8980
Batch 896/8980
Batch 928/8980
Batch 960/8980
Batch 992/8980
Batch 1024/8980
Batch 1056/8980
Batch 1088/8980
Batch 1120/8980
Batch 1152/8980
Batch 1184/8980
Batch 1216/8980
Batch 1248/8980
Batch 1280/8980
Batch 1312/8980
Batch 1344/8980
Batch 1376/8980
Batch 1408/8980
Batch 1440/8980
Batch 1472/8980
Batch 1504/8980
Batch 1536/8980
Batch 1568/8980
Batch 1600/8980
Batch 1632/8980
Batch 1664/8980
Batch 1696/8980
Batch 1728/8980
Batch 1760/8980
Batch 1792/8980
Batch 1824/8980
Batch 1856/8980
Batch 1888/8980
Batch 1920/8980
Batch 1952/8980
Batch 1984/8980
Batch 2016/8980
Batch 2048/89

In [ ]:
import os
print(os.listdir(f'{BASE}/preds'))

['lr_probs.npy', 'bert_probs.npy', 'lstm_probs.npy']


ensemble vote pondéré : LSTM + BERT +LR

In [ ]:
from sklearn.metrics import f1_score, roc_auc_score, classification_report
import joblib, pickle
import numpy as np

# Charger les prédictions LSTM et LR sauvegardées
#np.save(f'{BASE}/preds/bert_probs.npy', bert_probs)
bert_probs = np.load(f'{BASE}/preds/bert_probs.npy')

# Supposons qu'on a déjà sauvegardé ces arrays
lr_probs   = np.load(f'{BASE}/preds/lr_probs.npy')
lstm_probs = np.load(f'{BASE}/preds/lstm_probs.npy')

# Ensemble pondéré : BERT pèse plus car meilleur modèle
ensemble_probs = (0.6 * bert_probs +
                  0.25 * lstm_probs +
                  0.15 * lr_probs)

ensemble_preds = (ensemble_probs > 0.5).astype(int)

print("=== TABLEAU COMPARATIF FINAL ===")
models_results = {
    "TF-IDF + LR" : (lr_probs,   'baseline'),
    "BiLSTM+GloVe": (lstm_probs, 'deep'),
    "BERT"        : (bert_probs, 'transformer'),
    "Ensemble"    : (ensemble_probs, 'ensemble')
}
for name, (probs, _) in models_results.items():
    preds_ = (probs > 0.5).astype(int)
    print(f"{name:20s}  F1={f1_score(y_test,preds_,average='macro'):.4f}  "
          f"AUC={roc_auc_score(y_test,probs):.4f}")

=== TABLEAU COMPARATIF FINAL ===
TF-IDF + LR           F1=0.9823  AUC=0.9982
BiLSTM+GloVe          F1=0.9927  AUC=0.9996
BERT                  F1=0.0036  AUC=0.0002
Ensemble              F1=0.0036  AUC=0.0067


LIME : explication article par article

In [ ]:
!pip install lime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 22.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=8cdf2b6e2bb4a8129cd3d83910d7dc98e8a31f972830c391e3c3c7a9f509fe92
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime


In [ ]:
from lime.lime_text import LimeTextExplainer
import numpy as np

explainer_lime = LimeTextExplainer(class_names=['FAKE', 'REAL'])

#  modèle utilisé pour LIME (Logistic Regression)
def predict_for_lime(texts):
    X = tfidf.transform(texts)
    probs_real = lr.predict_proba(X)[:, 1]
    return np.column_stack([1 - probs_real, probs_real])

# choisir des erreurs du modèle ensemble
wrong_idx = np.where((ensemble_preds != y_test) & (y_test == 0))[0][:3]

for idx in wrong_idx:
    article = X_test.iloc[idx][:300]

    exp = explainer_lime.explain_instance(
        article,
        predict_for_lime,
        num_features=10,
        num_samples=200
    )

    print(f"\n--- Article #{idx} (FAKE mal classé REAL) ---")
    print(f"Texte : {article[:100]}...")
    print("Top features LIME :")

    for feat, weight in exp.as_list():
        direction = "→ FAKE" if weight > 0 else "→ REAL"
        print(f"{feat:20s} {weight:+.4f} {direction}")

# sauvegarde HTML
exp.save_to_file(f'{BASE}/figures/lime_explanation.html')


--- Article #2 (FAKE mal classé REAL) ---
Texte : democrat underbelly expose control violence erupt anti trump rioter deliver threat turn heat shut fr...
Top features LIME :
video                -0.1649 → REAL
wednesday            +0.0853 → FAKE
thug                 -0.0358 → REAL
police               +0.0339 → FAKE
clash                +0.0330 → FAKE
expose               -0.0312 → REAL
shut                 -0.0304 → REAL
anti                 -0.0281 → REAL
threat               -0.0217 → REAL
unit                 +0.0169 → FAKE

--- Article #5 (FAKE mal classé REAL) ---
Texte : sander supporter ready raise hell dnc mail leak prove dem steal election bernie wow week democrats l...
Top features LIME :
like                 -0.0369 → REAL
say                  +0.0343 → FAKE
hillary              -0.0263 → REAL
mail                 -0.0247 → REAL
dnc                  -0.0222 → REAL
wow                  -0.0188 → REAL
look                 -0.0181 → REAL
dem                  -0.0169 → REAL
el